In [2]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from hdbscan import HDBSCAN
import pickle
import umap



c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [39]:
df_new_aoi = pd.read_excel('./vec_clust_input_data.xlsx')

In [4]:
with open("./embeddings_new_aoi_back.pkl", "rb") as f:
    embeddings_new_aoi_back = pickle.load(f)
with open("./embeddings_new_aoi_front.pkl", "rb") as f:
    embeddings_new_aoi_front = pickle.load(f)

In [ ]:
len(embeddings_new_aoi_back)
len(embeddings_new_aoi_front)

<class 'numpy.ndarray'>


In [5]:
sim_matrix_embeddings_new_aoi_back = cosine_similarity(embeddings_new_aoi_back)
np.fill_diagonal(sim_matrix_embeddings_new_aoi_back, 0)
print("Max similarity_embeddings_new_aoi_back:", sim_matrix_embeddings_new_aoi_back.max())
print("Mean similarity_embeddings_new_aoi_back:", sim_matrix_embeddings_new_aoi_back.mean())
sim_matrix_embeddings_new_aoi_front = cosine_similarity(embeddings_new_aoi_front)
np.fill_diagonal(sim_matrix_embeddings_new_aoi_front, 0)
print("Max similarity_embeddings_new_aoi_front :", sim_matrix_embeddings_new_aoi_front.max())
print("Mean similarity_embeddings_new_aoi_front:", sim_matrix_embeddings_new_aoi_front.mean())

Max similarity_embeddings_new_aoi_back: 1.0000005
Mean similarity_embeddings_new_aoi_back: 0.8475823
Max similarity_embeddings_new_aoi_front : 1.0000007
Mean similarity_embeddings_new_aoi_front: 0.72094566


In [ ]:

X_std = StandardScaler().fit_transform(embeddings_new_aoi_front)       
X_2d  = umap.UMAP(n_components=3).fit_transform(X_std)  

<class 'numpy.ndarray'>


In [56]:
clusterer = HDBSCAN(min_cluster_size=15,min_samples=13,cluster_selection_method="leaf",algorithm="prims_kdtree")
labels = clusterer.fit_predict(X_std)
len(labels)
df_new_aoi["cluster_size_15_front_Scaled_min_sample_13_leaf_algo_prime_kdtree"] = labels

c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [58]:
df_new_aoi.to_excel("./euclidean_new_aoi_front.xlsx")

In [59]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# Step 1: Read your Excel file
input_excel = 'euclidean_new_aoi_front.xlsx' # Replace with actual input file path
df = pd.read_excel(input_excel)  # Reads first sheet or specify sheet_name='Sheet1'

# Step 2: Find all cluster columns starting with "cluster_"
cluster_cols = [col for col in df.columns if col.startswith("cluster_")]

print(f"Cluster columns found: {cluster_cols}")

# Step 3: Prepare output Excel file
output_excel = "cluster_sizes_single_sheet_with_headers_front.xlsx"

# Step 4: Create an Excel file with an empty sheet named 'Clusters'
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    pd.DataFrame().to_excel(writer, sheet_name='Clusters', index=False)

# Step 5: Open workbook and set active sheet
wb = load_workbook(output_excel)
ws = wb['Clusters']

# Step 6: Write all cluster sizes tables sequentially with headers
start_row = 1
for col_name in cluster_cols:
    labels = df[col_name]
    cluster_sizes = labels.value_counts().sort_index()

    # Prepare clearer labels for noise
    cluster_labels = ['Noise (-1)' if x == -1 else x for x in cluster_sizes.index]

    # Write cluster column name as bold merged header across two columns
    ws.cell(row=start_row, column=1, value=col_name)
    ws.cell(row=start_row, column=1).font = Font(bold=True)
    ws.cell(row=start_row, column=1).alignment = Alignment(horizontal='center')
    ws.merge_cells(start_row=start_row, start_column=1, end_row=start_row, end_column=2)

    # Write table header (Cluster Label, Number of Points)
    ws.cell(row=start_row+1, column=1, value='Cluster Label').font = Font(bold=True)
    ws.cell(row=start_row+1, column=2, value='Number of Points').font = Font(bold=True)

    # Write cluster size data below header
    for i, (label, count) in enumerate(zip(cluster_labels, cluster_sizes.values), start=start_row+2):
        ws.cell(row=i, column=1, value=label)
        ws.cell(row=i, column=2, value=count)

    # Update start_row for next cluster table (2 header rows + number of clusters + 2 blank rows)
    start_row = start_row + 2 + len(cluster_sizes) + 2

# Optional: Adjust column widths for readability
for col_idx in [1, 2]:
    col_letter = get_column_letter(col_idx)
    max_length = max(len(str(cell.value)) for cell in ws[col_letter] if cell.value)
    ws.column_dimensions[col_letter].width = max_length + 2

# Step 7: Save the updated Excel file
wb.save(output_excel)
print(f"All cluster size summaries with headers written to '{output_excel}'.")


Cluster columns found: ['cluster_size_15_front_3d_min_sample_13_eom', 'cluster_size_15_front_3d_min_sample_13_leaf', 'cluster_size_15_front_Scaled_min_sample_13_leaf', 'cluster_size_15_front_Scaled_min_sample_13_leaf_algo_prime_kdtree']
All cluster size summaries with headers written to 'cluster_sizes_single_sheet_with_headers_front.xlsx'.


In [62]:
# First, install pyclustering if not already installed:
# pip install pyclustering

from pyclustering.cluster.xmeans import xmeans
from pyclustering.cluster.center_initializer import kmeans_plusplus_initializer
from pyclustering.cluster import cluster_visualizer

# Example data: list of points with two features each
# Replace this with your actual data, e.g. a list of lists or numpy array
data = [
    [1.0, 2.0], [1.1, 2.1], [0.9, 1.8],
    [5.0, 8.0], [5.1, 7.9], [4.9, 8.2],
    [20.0, 25.0], [21.0, 24.5], [19.5, 26.0]
]

# Step 1: Initialize starting centers with kmeans++ 
# (number is initial clusters to try, usually small)
initial_centers = kmeans_plusplus_initializer(data, 2).initialize()

# Step 2: Create X-Means instance with max clusters = 10 for example
xmeans_instance = xmeans(data, initial_centers, kmax=10)

# Step 3: Run the algorithm
xmeans_instance.process()

# Step 4: Extract results
clusters = xmeans_instance.get_clusters()
centers = xmeans_instance.get_centers()

print("Clusters (point indices):", clusters)
print("Centers:", centers)

# Optional: assign cluster labels to each data point
labels = [0] * len(data)
for cluster_id, cluster_points in enumerate(clusters):
    for index in cluster_points:
        labels[index] = cluster_id

print("Cluster labels for data points:", labels)

# Optional: Visualize the clustering result (2D data only)
visualizer = cluster_visualizer()
visualizer.append_clusters(clusters, data)
visualizer.append_cluster(centers, None, marker='*', markersize=10)
visualizer.show()


AttributeError: module 'numpy' has no attribute 'warnings'